# How Many Generations Does Relatedness-Driven Similarity Last? (vem 1uM, 144hrs Pre-Treatment)

**Prototype.** `01_relatedness.ipynb` found that sisters (degree 1: common parent) and cousins (degree 2: common grandparent) are both more morphologically similar than random, for both modalities and both the control and treatment ranges. This notebook asks the natural follow-up: how far does that similarity reach - does it still hold for second cousins (degree 3: common great-grandparent), third cousins (degree 4), and so on, or does it fade out after a few generations?

`shared/lineage_tree.py`'s `find_relative_pairs(segments, degree)` generalizes `find_sister_pairs`/`find_cousin_pairs` to any `degree` (1 = sisters, 2 = cousins, 3 = second cousins, ...). For each `degree`, this notebook runs the exact same shared-null permutation test as `01_relatedness.ipynb`, once per (modality, range) - **4 lines total on one plot**: x-axis is `degree`, y-axis is `-log10(p-value)` (higher = more significant), with a horizontal line at `p = 0.05` so each line's crossing point is visible directly.

**The real constraint is sample size, not the method.** The number of degree-`d` pairs whose BOTH sides fall inside a 5-frame window shrinks fast as `d` grows - especially under treatment, where persisters/dying cells barely divide. A line stops because it ran out of qualifying pairs (printed, and visible as the line simply ending) is a *different* conclusion from a line that keeps going but crosses above `p = 0.05` (relatedness-driven similarity genuinely fades) - the printed `n_pairs` per point is what tells these apart, so read it alongside the plot rather than just the crossing point.

Requires `main.py` to have already been run (for `selected_metadata.csv` and `{cell,nuc}_qgw_075P_dists.csv` under `output/cajal_output/`).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/05_vem_144hr_lineage_relatedness/notebooks/), so the analysis
# dir - and config.py - is one level up, and the repo root - where
# shared/ lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import parse_label
from shared.lineage_tree import build_segments, find_relative_pairs, real_index_pairs, pool_in_range
from shared.pair_similarity import pair_distance_permutation_test

## Method

1. **Reconstruct every lineage tree** once (same as `01_relatedness.ipynb`, from the FULL metadata in frames 1-`config.TREATMENT_END_FRAME`, not just the CAJAL-sampled cells).
2. **For each modality** (cell, nucleus), load its real distance matrix once.
3. **For `degree` = 1, 2, 3, ...**: find every degree-`degree` relative pair across every tree (`find_relative_pairs`) - this step doesn't depend on modality or frame range, so it's done once per degree, not once per (modality, range). Then, separately for each (modality, range):
   - Restrict to whichever of each pair's own observations fall in that range AND survived CAJAL subsampling (`real_index_pairs`, exactly as in `01_relatedness.ipynb`) - every surviving combination counts as its own pair, not just one per relationship.
   - If this (modality, range) has zero surviving pairs at this degree, it has run out of data - stop extending that one line (the others may continue).
   - Otherwise, run the shared-null permutation test (`pair_distance_permutation_test`) against every CAJAL-embedded cell in that same range, using the pairs' own count as the draw size (unlike `01_relatedness.ipynb`, which calibrates sisters and cousins to a shared draw size so they share one null histogram - there's no such need here, since each `degree` gets its own point on this line plot, not an overlaid histogram).
4. **Plot**: one line per (modality, range) - 4 total - of `-log10(p-value)` vs. `degree`, with a horizontal line at `p = 0.05`.

In [ ]:
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '02_relatedness_decay')
SVG_DIR = os.path.join(PLOTS_DIR, 'svgs')
os.makedirs(SVG_DIR, exist_ok=True)

MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}
MODALITY_COLORS = {'cell': 'tab:blue', 'nuc': 'tab:orange'}
RANGE_LINESTYLES = {'control': '-', 'treatment': '--'}

# same (start, end) frame ranges as 01_relatedness.ipynb
FRAME_RANGES = {
    'control':   (1, config.CONTROL_END_FRAME),
    'treatment': (config.TREATMENT_START_FRAME, config.TREATMENT_END_FRAME),
}

# a safety cap, not expected to actually bind - find_relative_pairs
# itself (independent of modality/range) is what actually stops the
# sweep in practice, as soon as NO tree anywhere has divided this many
# times yet; this just guards against an unexpectedly deep tree
MAX_DEGREE = 10

# below this many surviving pairs, a point's p-value is too noisy to
# read as a real trend - flagged in the printed summary, not hidden
MIN_RELIABLE_PAIRS = 5

## Build the lineage tree and load each modality's distance matrix (once)

In [ ]:
full_metadata = pd.read_csv(config.metadata_csv_path())

# tree-building uses the FULL metadata (every tracked cell, has_mask
# True or False) - has_mask only reflects whether a usable mask crop
# could be extracted, not whether the cell/lineage tracking itself
# exists. Filtering to has_mask=True here would drop intermediate rows
# a later frame's pred_node_id still needs, breaking chains that are
# otherwise intact.
full_metadata = full_metadata[full_metadata['frame'] <= config.TREATMENT_END_FRAME].reset_index(drop=True)

segments = build_segments(full_metadata)
print(f"Built {len(segments)} segment(s) across every lineage tree in frames 1-{config.TREATMENT_END_FRAME}")

modality_data = {}
for modality in ['cell', 'nuc']:
    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    labels, dist_matrix = load_distance_matrix(dist_csv)
    label_to_index = {}
    for i, label in enumerate(labels):
        _, frame, pixel_id = parse_label(label)
        label_to_index[(frame, pixel_id)] = i

    modality_data[modality] = {
        'dist_matrix': dist_matrix,
        'label_to_index': label_to_index,
        # the null's population is fixed per (modality, range) - it
        # does NOT shrink with degree, only the real pairs being
        #  tested against it do
        'pool': {range_name: pool_in_range(label_to_index, frame_range) for range_name, frame_range in FRAME_RANGES.items()},
    }
    print(f"[{modality}] {len(labels)} CAJAL-embedded cell(s) total")

## Sweep: degree = 1, 2, 3, ... for each (modality, range)

In [ ]:
results = []
combos = [(modality, range_name) for modality in ['cell', 'nuc'] for range_name in FRAME_RANGES]
active = {combo: True for combo in combos}

for degree in range(1, MAX_DEGREE + 1):
    topology_pairs = find_relative_pairs(segments, degree)
    if not topology_pairs:
        print(f"degree {degree}: no lineage anywhere has divided this many times yet - stopping every line")
        break

    for modality, range_name in combos:
        if not active[(modality, range_name)]:
            continue

        data = modality_data[modality]
        frame_range = FRAME_RANGES[range_name]
        real_idx = real_index_pairs(topology_pairs, frame_range, data['label_to_index'])

        if len(real_idx) == 0:
            active[(modality, range_name)] = False
            print(f"  [{modality}/{range_name}] degree {degree}: 0 surviving pairs - stopping this line")
            continue

        mean_distance, null_means, p_value = pair_distance_permutation_test(
            data['dist_matrix'], data['pool'][range_name], real_idx, len(real_idx),
            n_permutations=config.N_PERMUTATIONS, seed=config.TEST_SEED,
        )
        flag = '  <-- fewer than MIN_RELIABLE_PAIRS, read with caution' if len(real_idx) < MIN_RELIABLE_PAIRS else ''
        print(f"  [{modality}/{range_name}] degree {degree}: {len(real_idx)} pair(s), "
              f"mean distance = {mean_distance:.4f}, p = {p_value:.4f}{flag}")

        results.append({
            'modality': modality, 'frame_range': range_name, 'degree': degree,
            'n_pairs': len(real_idx), 'mean_distance': mean_distance, 'p_value': p_value,
        })

    if not any(active.values()):
        break

results_df = pd.DataFrame(results)
print(f"\n{len(results_df)} (modality, range, degree) point(s) total")

## Plot: significance vs. relatedness degree, all 4 (modality, range) combinations on one plot

No error bars - see the discussion in chat for why: a per-pair p-value distribution and a bootstrap/Clopper-Pearson CI were both explored and set aside (the former as a separate diagnostic to build later if wanted, the latter over implementation issues not worth the added complexity here). A hollow marker instead flags any point where the permutation test hit its own resolution floor (`p_value == 0`: no random draw beat the real pairs at all out of `n_permutations` tries) - there's no floor on how small the TRUE value could be in that case, only a floor on how confidently this test can tell, so that point's exact height is a floor-clipped stand-in, not a precise measurement.

In [ ]:
P_FLOOR = 1 / config.N_PERMUTATIONS
results_df['at_resolution_floor'] = results_df['p_value'] == 0
results_df['neg_log10_p'] = -np.log10(results_df['p_value'].clip(lower=P_FLOOR))

# a small, fixed per-(modality, range) horizontal offset so points that
# land on the exact same (degree, y) - common near the resolution
# floor, where several series all hit p == 0 at once - don't sit
# exactly on top of each other
DODGE = {
    ('cell', 'control'):   -0.09,
    ('cell', 'treatment'): -0.03,
    ('nuc',  'control'):    0.03,
    ('nuc',  'treatment'):  0.09,
}

fig, ax = plt.subplots(figsize=(8, 5.5))

for (modality, range_name), group in results_df.groupby(['modality', 'frame_range']):
    group = group.sort_values('degree')
    degree = group['degree'].to_numpy() + DODGE[(modality, range_name)]
    ax.plot(
        degree, group['neg_log10_p'],
        color=MODALITY_COLORS[modality], linestyle=RANGE_LINESTYLES[range_name], marker='o',
        label=f'{MODALITY_NAMES[modality]}, {range_name}',
    )
    floored = group[group['at_resolution_floor']]
    if not floored.empty:
        ax.scatter(
            floored['degree'].to_numpy() + DODGE[(modality, range_name)], floored['neg_log10_p'],
            facecolors='none', edgecolors=MODALITY_COLORS[modality], s=140, linewidths=1.5, zorder=5,
        )

threshold = -np.log10(0.05)
ax.axhline(threshold, color='grey', linestyle=':', linewidth=1.5)
ax.text(results_df['degree'].max() + 0.15, threshold, '  p = 0.05', va='center', ha='left', color='grey')

ax.set_xlabel('relatedness degree (1 = sisters, 2 = cousins, 3 = second cousins, ...)')
ax.set_ylabel('-log10(p-value)\n(higher = more significantly more similar than a random pair)')
ax.set_xticks(sorted(results_df['degree'].unique()))
ax.set_title("How far does relatedness-driven similarity reach?\n(hollow marker = p below this test's resolution limit)")
ax.legend()
fig.tight_layout()

fig.savefig(os.path.join(PLOTS_DIR, 'relatedness_decay.pdf'), bbox_inches='tight')
fig.savefig(os.path.join(SVG_DIR, 'relatedness_decay.svg'), bbox_inches='tight')
display(fig)   # show it in the notebook's own output, not just save it to disk
plt.close(fig)
print(f"Saved to {PLOTS_DIR}")

## Where does each line cross p = 0.05?

The first degree, per (modality, range), where `p_value >= 0.05` - i.e. where relatedness-driven similarity stops being distinguishable from chance. A combo that ran out of pairs before ever crossing is reported separately: that's "not enough data to tell", not "similarity never fades".

In [ ]:
for modality, range_name in combos:
    group = results_df[(results_df['modality'] == modality) & (results_df['frame_range'] == range_name)].sort_values('degree')
    label = f'{MODALITY_NAMES[modality]}, {range_name}'
    if group.empty:
        print(f"{label}: no degree had any surviving pairs at all")
        continue

    lost = group[group['p_value'] >= 0.05]
    max_degree_reached = group['degree'].max()
    ran_out = not active[(modality, range_name)]

    if not lost.empty:
        crossing_degree = int(lost['degree'].iloc[0])
        n_pairs = int(lost['n_pairs'].iloc[0])
        print(f"{label}: crosses p = 0.05 at degree {crossing_degree} (n_pairs = {n_pairs})")
    elif ran_out:
        print(f"{label}: still significant through degree {max_degree_reached}, but ran out of pairs before crossing - "
              f"not enough data to say whether it would eventually cross")
    else:
        print(f"{label}: still significant through degree {max_degree_reached} (the sweep's own cap) - never crossed")

## Export plot data

One row per (modality, frame_range, degree): `n_pairs`, `mean_distance`, `p_value`, `at_resolution_floor`, `neg_log10_p` - everything needed to redraw the plot above or re-derive the crossing points, without rerunning the permutation tests.

In [ ]:
plot_data_path = os.path.join(PLOTS_DIR, 'plot_data.csv')
results_df.to_csv(plot_data_path, index=False)
print(f"Saved plot data ({len(results_df)} rows) to {plot_data_path}")